# Phase 1: Masking & Demarcation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/forest-carbon-pipeline/blob/main/notebooks/01_Masking_Demarcation.ipynb)

Dynamic NDVI/SAVI thresholding (L=0.5) to isolate forest pixels.


In [ ]:
# Setup: locate the repo's `src` package (any cwd inside the repo, or fresh clone)
import os
import subprocess
import sys

REPO = "forest-carbon-pipeline"
if os.path.isdir("src"):
    ROOT = os.getcwd()
elif os.path.isdir(os.path.join("..", "src")):
    ROOT = os.path.abspath("..")
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline.git"],
                   check=False)
    ROOT = os.path.abspath(REPO)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
if not os.path.isdir(os.path.join(ROOT, "src")):
    raise RuntimeError(
        f"Could not locate the pipeline source at {ROOT}. Open this notebook "
        "from a repo checkout, or check your network so the bootstrap can "
        "clone https://github.com/SyedRafiAhmedGIS/forest-carbon-pipeline"
    )

from src.phase1_demarcate import demarcate_forest_pixels  # noqa: E402
QUARANTINE = os.path.join(ROOT, "quarantine")
print(f"Pipeline package ready (ROOT = {ROOT})")


## Synthetic scene

100 x 100 px at 20 m, seeded so every re-run is byte-identical.


In [ ]:
# Synthetic Sentinel-2-like scene: 100 x 100 px at 20 m (deterministic)
import numpy as np

np.random.seed(42)
nir_band = np.random.uniform(0.2, 0.8, (100, 100))
red_band = np.random.uniform(0.05, 0.3, (100, 100))
savi = ((nir_band - red_band) / (nir_band + red_band + 0.5)) * 1.5  # L = 0.5
print("Scene ready: 100x100 px @ 20 m, seed=42")


## NDVI & SAVI formulas

- NDVI = (NIR - Red) / (NIR + Red)
- SAVI = ((NIR - Red) / (NIR + Red + L)) x (1 + L), with L = 0.5

Pixels with NDVI > 0.3 are demarcated as forest. Rasters that are more than 50% NoData are routed to the quarantine log and never reach Phase 2.


In [ ]:
forest_mask, ndvi, input_hash = demarcate_forest_pixels(
    nir_band, red_band, quarantine_dir=QUARANTINE)


### What just happened

- `forest_mask` - boolean forest / non-forest
- `ndvi` - continuous index used downstream
- `input_hash` - MD5 of the input bands, embedded in Phase 6 provenance metadata; re-running the same scene reproduces the hash byte for byte.
